# Objective 3 — Step 4: Group-Aware Feature-Selection Development on German Credit

This notebook is the **development stage** for the Objective-3 feature-selection component.

## Why only German Credit in Step 4?
German Credit is treated as the development dataset. We will select the feature-selection design and subset proportion **without looking at Australian or Taiwan performance**. After reviewing this output, the chosen design will be frozen and transferred to the other datasets in the next stage.

## Methods
Five established selectors are evaluated:

1. Chi-Square
2. Mutual Information
3. L1-regularised Logistic Regression
4. Random-Forest importance
5. Recursive Feature Elimination (RFE)

A sixth candidate is a **group-aware equal consensus** of all five rankings.

## Group-aware improvement
Feature selection is performed at the **original/source-variable level**, not at individual one-hot dummy level. Scores from encoded columns belonging to the same original variable are aggregated before final ranking. This directly improves interpretability and prevents partial selection of a categorical variable.

## Validation
- Same German outer folds as Step 3.
- 5 folds × 5 repetitions = 25 paired runs.
- Every selector is fitted on the outer-training partition only.
- XGBoost is used as the fixed reference classifier because Step 3 showed the highest German ROC-AUC and strongest overall German baseline.
- No class weighting, SMOTE, calibration, threshold optimisation, or hyperparameter tuning is used here.

**Do not inspect Australian or Taiwan performance when selecting the Step-4 winner.**


In [1]:
%pip install pandas numpy scikit-learn xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:

from pathlib import Path
from itertools import combinations
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import chi2, mutual_info_classif, RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

STEP2_DIR = BASE_DIR / "results" / "preprocessing_protocol"
BASELINE_DIR = BASE_DIR / "results" / "baseline_models"
DATA_DIR = BASE_DIR / "data" / "processed"

OUT_DIR = BASE_DIR / "results" / "feature_selection_german_development"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GERMAN_FILE = DATA_DIR / "german_credit_cleaned.csv"
DICTIONARY_FILE = STEP2_DIR / "preprocessing_data_dictionary.csv"
BASELINE_RESULTS_FILE = BASELINE_DIR / "baseline_fold_results_all.csv"
BASELINE_PREDICTIONS_FILE = BASELINE_DIR / "baseline_predictions_all.csv"

required = [
    GERMAN_FILE,
    DICTIONARY_FILE,
    BASELINE_RESULTS_FILE,
    BASELINE_PREDICTIONS_FILE,
]

missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Required previous-step files are missing:\n"
        + "\n".join(missing)
    )

RANDOM_STATE = 42
REPEAT_SEEDS = [42, 142, 242, 342, 442]
N_FOLDS = 5

SUBSET_FRACTIONS = {
    "Top25": 0.25,
    "Top50": 0.50,
    "Top75": 0.75,
}

SELECTOR_NAMES = [
    "Chi2",
    "MI",
    "L1",
    "RF",
    "RFE",
    "EqualConsensus",
]

print("Output folder:", OUT_DIR)


Output folder: D:\PHD\Research Paper writing\3rd Obj. paper\results\feature_selection_german_development


## 1. Load German Credit and feature roles

In [3]:

df = pd.read_csv(GERMAN_FILE)
dictionary = pd.read_csv(DICTIONARY_FILE)
baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)
baseline_predictions = pd.read_csv(BASELINE_PREDICTIONS_FILE)

dataset_name = "German Credit"

d = dictionary[dictionary["dataset"] == dataset_name].copy()

categorical_features = (
    d.loc[d["role"] == "categorical", "variable"]
    .astype(str)
    .tolist()
)

ordinal_features = (
    d.loc[d["role"] == "ordinal", "variable"]
    .astype(str)
    .tolist()
)

numerical_features = (
    d.loc[d["role"] == "numerical", "variable"]
    .astype(str)
    .tolist()
)

source_features = (
    categorical_features
    + ordinal_features
    + numerical_features
)

X = df[source_features].copy()
y = df["adverse_target"].astype(int).copy()
groups = df["profile_group_id"].astype(str).copy()

print("Records:", len(df))
print("Source predictors:", len(source_features))
print("Categorical:", len(categorical_features))
print("Ordinal:", len(ordinal_features))
print("Numerical:", len(numerical_features))
print("Adverse rate:", round(y.mean(), 4))
print("\nSource features:")
print(source_features)

assert len(source_features) == 20
assert set(y.unique()).issubset({0, 1})


Records: 1000
Source predictors: 20
Categorical: 13
Ordinal: 0
Numerical: 7
Adverse rate: 0.3

Source features:
['status', 'credit_history', 'purpose', 'savings', 'employment_duration', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker', 'duration', 'amount', 'installment_rate', 'present_residence', 'age', 'number_credits', 'people_liable']


## 2. Recreate and verify exactly the same Step-3 outer folds

In [4]:

def generate_outer_splits():
    split_dict = {}

    for repeat_no, seed in enumerate(REPEAT_SEEDS, start=1):
        splitter = StratifiedGroupKFold(
            n_splits=N_FOLDS,
            shuffle=True,
            random_state=seed,
        )

        for fold_no, (train_idx, test_idx) in enumerate(
            splitter.split(X, y, groups),
            start=1,
        ):
            run_id = f"R{repeat_no}_F{fold_no}"

            train_groups = set(groups.iloc[train_idx])
            test_groups = set(groups.iloc[test_idx])

            assert len(
                train_groups.intersection(test_groups)
            ) == 0

            split_dict[run_id] = {
                "repeat": repeat_no,
                "fold": fold_no,
                "seed": seed,
                "train_idx": np.asarray(train_idx, dtype=int),
                "test_idx": np.asarray(test_idx, dtype=int),
            }

    return split_dict


OUTER_SPLITS = generate_outer_splits()

# Strong verification against Step-3 saved XGBoost predictions.
step3_xgb_predictions = baseline_predictions[
    (baseline_predictions["dataset"] == dataset_name)
    & (baseline_predictions["model"] == "XGB")
].copy()

for run_id, info in OUTER_SPLITS.items():
    expected = set(info["test_idx"].tolist())

    saved = set(
        step3_xgb_predictions.loc[
            step3_xgb_predictions["run_id"] == run_id,
            "source_row_index",
        ].astype(int).tolist()
    )

    assert expected == saved, (
        f"Step-4 split {run_id} does not match Step 3."
    )

print("All 25 Step-4 outer test folds exactly match Step 3.")


All 25 Step-4 outer test folds exactly match Step 3.


## 3. Build preprocessing variants

In [5]:

def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )


def build_preprocessor(mode, selected_source_features=None):
    if selected_source_features is None:
        selected_source_features = source_features

    selected_source_features = list(selected_source_features)

    selected_cat = [
        feature
        for feature in categorical_features
        if feature in selected_source_features
    ]

    selected_ord = [
        feature
        for feature in ordinal_features
        if feature in selected_source_features
    ]

    selected_num = [
        feature
        for feature in numerical_features
        if feature in selected_source_features
    ]

    transformers = []

    if selected_num:
        if mode == "scaled":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "chi2":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        elif mode == "tree":
            num_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ])
        else:
            raise ValueError(mode)

        transformers.append(
            ("num", num_pipe, selected_num)
        )

    if selected_ord:
        if mode == "scaled":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", StandardScaler()),
            ])
        elif mode == "chi2":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("scaler", MinMaxScaler(clip=True)),
            ])
        elif mode == "tree":
            ord_pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
            ])
        else:
            raise ValueError(mode)

        transformers.append(
            ("ord", ord_pipe, selected_ord)
        )

    if selected_cat:
        cat_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])

        transformers.append(
            ("cat", cat_pipe, selected_cat)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        verbose_feature_names_out=True,
    )


## 4. Dynamic transformed-feature mapping

In [6]:

def feature_map_from_fitted_preprocessor(
    fitted_preprocessor,
    selected_source_features=None,
):
    if selected_source_features is None:
        selected_source_features = source_features

    selected_source_features = list(selected_source_features)

    selected_cat = [
        f for f in categorical_features
        if f in selected_source_features
    ]
    selected_ord = [
        f for f in ordinal_features
        if f in selected_source_features
    ]
    selected_num = [
        f for f in numerical_features
        if f in selected_source_features
    ]

    rows = []
    transformed_index = 0

    for feature in selected_num:
        rows.append({
            "transformed_index": transformed_index,
            "transformed_feature": f"num__{feature}",
            "source_feature": feature,
            "source_role": "numerical",
        })
        transformed_index += 1

    for feature in selected_ord:
        rows.append({
            "transformed_index": transformed_index,
            "transformed_feature": f"ord__{feature}",
            "source_feature": feature,
            "source_role": "ordinal",
        })
        transformed_index += 1

    if selected_cat:
        cat_pipeline = fitted_preprocessor.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["onehot"]

        encoded_names = encoder.get_feature_names_out(selected_cat)

        name_position = 0

        for source_feature, categories in zip(
            selected_cat,
            encoder.categories_,
        ):
            for _ in categories:
                rows.append({
                    "transformed_index": transformed_index,
                    "transformed_feature": (
                        "cat__" + str(encoded_names[name_position])
                    ),
                    "source_feature": source_feature,
                    "source_role": "categorical",
                })

                transformed_index += 1
                name_position += 1

    feature_map = pd.DataFrame(rows)

    assert len(feature_map) == len(
        fitted_preprocessor.get_feature_names_out()
    )

    return feature_map


## 5. Convert transformed-variable evidence into source-feature scores

For every selector:

1. transformed-variable scores are ranked within the training fold;
2. ranks are converted to normalized relevance scores from 0 to 1;
3. encoded columns belonging to the same original feature are aggregated by their **mean normalized relevance**;
4. source features are ranked again.

Using the mean avoids mechanically rewarding categorical variables merely because they contain more one-hot columns.


In [7]:

def transformed_scores_to_source_scores(
    raw_scores,
    feature_map,
    selector_name,
):
    temp = feature_map.copy()
    temp["raw_score"] = np.asarray(raw_scores, dtype=float)

    temp["raw_score"] = (
        temp["raw_score"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    n = len(temp)

    transformed_rank = temp["raw_score"].rank(
        ascending=False,
        method="average",
    )

    if n > 1:
        temp["normalized_relevance"] = (
            1.0 - (transformed_rank - 1.0) / (n - 1.0)
        )
    else:
        temp["normalized_relevance"] = 1.0

    grouped = (
        temp.groupby("source_feature", as_index=False)
        .agg(
            group_score=("normalized_relevance", "mean"),
            transformed_columns=("transformed_feature", "count"),
        )
    )

    p = len(grouped)

    grouped["source_rank"] = grouped["group_score"].rank(
        ascending=False,
        method="average",
    )

    if p > 1:
        grouped["source_normalized_score"] = (
            1.0
            - (grouped["source_rank"] - 1.0) / (p - 1.0)
        )
    else:
        grouped["source_normalized_score"] = 1.0

    grouped["selector"] = selector_name

    return grouped.sort_values(
        ["source_rank", "source_feature"]
    ).reset_index(drop=True)


## 6. Fit the five selectors within one outer-training fold

In [8]:

def selector_rankings_for_training_fold(
    X_train,
    y_train,
    fold_seed,
):
    # Three independent fold-fitted representations.
    prep_chi2 = build_preprocessor("chi2")
    prep_scaled = build_preprocessor("scaled")
    prep_tree = build_preprocessor("tree")

    X_chi2 = prep_chi2.fit_transform(
        X_train,
        y_train,
    )
    X_scaled = prep_scaled.fit_transform(
        X_train,
        y_train,
    )
    X_tree = prep_tree.fit_transform(
        X_train,
        y_train,
    )

    map_chi2 = feature_map_from_fitted_preprocessor(
        prep_chi2
    )
    map_scaled = feature_map_from_fitted_preprocessor(
        prep_scaled
    )
    map_tree = feature_map_from_fitted_preprocessor(
        prep_tree
    )

    # -----------------
    # Chi-Square
    # -----------------
    chi_scores, _ = chi2(
        X_chi2,
        y_train,
    )

    chi_group = transformed_scores_to_source_scores(
        chi_scores,
        map_chi2,
        "Chi2",
    )

    # -----------------
    # Mutual Information
    # -----------------
    discrete_mask = (
        map_tree["source_role"]
        .eq("categorical")
        .to_numpy()
    )

    mi_scores = mutual_info_classif(
        X_tree,
        y_train,
        discrete_features=discrete_mask,
        random_state=fold_seed,
    )

    mi_group = transformed_scores_to_source_scores(
        mi_scores,
        map_tree,
        "MI",
    )

    # -----------------
    # L1 Logistic Regression
    # -----------------
    l1_model = LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=1.0,
        max_iter=3000,
        random_state=fold_seed,
    )

    l1_model.fit(
        X_scaled,
        y_train,
    )

    l1_scores = np.abs(
        l1_model.coef_.ravel()
    )

    l1_group = transformed_scores_to_source_scores(
        l1_scores,
        map_scaled,
        "L1",
    )

    # -----------------
    # Random-Forest importance
    # -----------------
    rf_selector = RandomForestClassifier(
        n_estimators=300,
        random_state=fold_seed,
        n_jobs=-1,
    )

    rf_selector.fit(
        X_tree,
        y_train,
    )

    rf_scores = rf_selector.feature_importances_

    rf_group = transformed_scores_to_source_scores(
        rf_scores,
        map_tree,
        "RF",
    )

    # -----------------
    # RFE with Logistic Regression
    # -----------------
    rfe_estimator = LogisticRegression(
        solver="liblinear",
        max_iter=3000,
        random_state=fold_seed,
    )

    rfe = RFE(
        estimator=rfe_estimator,
        n_features_to_select=1,
        step=0.10,
    )

    rfe.fit(
        X_scaled,
        y_train,
    )

    # Smaller RFE rank means stronger feature.
    rfe_scores = 1.0 / np.asarray(
        rfe.ranking_,
        dtype=float,
    )

    rfe_group = transformed_scores_to_source_scores(
        rfe_scores,
        map_scaled,
        "RFE",
    )

    individual = pd.concat(
        [
            chi_group,
            mi_group,
            l1_group,
            rf_group,
            rfe_group,
        ],
        ignore_index=True,
    )

    # Equal consensus over source-level normalized rankings.
    consensus = (
        individual
        .pivot_table(
            index="source_feature",
            columns="selector",
            values="source_normalized_score",
            aggfunc="mean",
        )
        .reset_index()
    )

    method_columns = [
        "Chi2", "MI", "L1", "RF", "RFE"
    ]

    consensus["group_score"] = consensus[
        method_columns
    ].mean(axis=1)

    consensus["source_rank"] = consensus[
        "group_score"
    ].rank(
        ascending=False,
        method="average",
    )

    p = len(consensus)

    consensus["source_normalized_score"] = (
        1.0
        - (consensus["source_rank"] - 1.0)
        / (p - 1.0)
    )

    consensus["transformed_columns"] = np.nan
    consensus["selector"] = "EqualConsensus"

    consensus = consensus[
        [
            "source_feature",
            "group_score",
            "transformed_columns",
            "source_rank",
            "source_normalized_score",
            "selector",
        ]
    ]

    all_rankings = pd.concat(
        [individual, consensus],
        ignore_index=True,
    )

    return all_rankings


## 7. Fixed XGBoost evaluation model

In [9]:

def fresh_xgb():
    return XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )


def calculate_metrics(
    y_true,
    y_pred,
    y_score,
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    gmean = (
        math.sqrt(sensitivity * specificity)
        if not np.isnan(
            sensitivity + specificity
        )
        else np.nan
    )

    fpr, tpr, _ = roc_curve(
        y_true,
        y_score,
    )

    ks = float(
        np.max(tpr - fpr)
    )

    return {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "precision_adverse": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "recall_adverse": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "specificity": specificity,
        "f1_adverse": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "mcc": matthews_corrcoef(
            y_true,
            y_pred,
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_score,
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_score,
        ),
        "gmean": gmean,
        "ks_statistic": ks,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


## 8. Run German feature-selection development

This cell evaluates:

- 6 ranking methods
- 3 subset proportions
- 25 paired outer folds

That produces **450 feature-selected XGBoost evaluations**.

The selectors themselves are fitted only once per outer training fold and reused for the three subset sizes.


In [10]:

ranking_rows = []
selection_rows = []
performance_rows = []

p = len(source_features)

for run_number, (
    run_id,
    split_info,
) in enumerate(
    OUTER_SPLITS.items(),
    start=1,
):
    print(
        f"\n{'='*70}\n"
        f"{run_id} ({run_number}/25)\n"
        f"{'='*70}"
    )

    train_idx = split_info["train_idx"]
    test_idx = split_info["test_idx"]

    X_train = X.iloc[train_idx].copy()
    X_test = X.iloc[test_idx].copy()
    y_train = y.iloc[train_idx].copy()
    y_test = y.iloc[test_idx].copy()

    selector_start = time.perf_counter()

    rankings = selector_rankings_for_training_fold(
        X_train,
        y_train,
        fold_seed=split_info["seed"],
    )

    selector_runtime = (
        time.perf_counter()
        - selector_start
    )

    rankings["run_id"] = run_id
    rankings["repeat"] = split_info["repeat"]
    rankings["fold"] = split_info["fold"]

    ranking_rows.extend(
        rankings.to_dict("records")
    )

    print(
        f"Selector rankings completed "
        f"in {selector_runtime:.2f}s"
    )

    for selector_name in SELECTOR_NAMES:
        selector_ranking = (
            rankings[
                rankings["selector"]
                == selector_name
            ]
            .sort_values(
                [
                    "source_rank",
                    "source_feature",
                ]
            )
            .copy()
        )

        ordered_features = (
            selector_ranking[
                "source_feature"
            ]
            .astype(str)
            .tolist()
        )

        assert len(ordered_features) == p
        assert len(set(ordered_features)) == p

        for subset_label, fraction in (
            SUBSET_FRACTIONS.items()
        ):
            n_select = int(
                math.ceil(
                    p * fraction
                )
            )

            selected = ordered_features[
                :n_select
            ]

            reduction_pct = (
                100.0
                * (1.0 - n_select / p)
            )

            selection_rows.append({
                "dataset": dataset_name,
                "run_id": run_id,
                "repeat": split_info["repeat"],
                "fold": split_info["fold"],
                "selector": selector_name,
                "subset": subset_label,
                "subset_fraction": fraction,
                "selected_source_features": n_select,
                "total_source_features": p,
                "feature_reduction_pct": reduction_pct,
                "selected_features": ";".join(selected),
            })

            eval_preprocessor = (
                build_preprocessor(
                    "tree",
                    selected_source_features=selected,
                )
            )

            eval_pipeline = Pipeline([
                (
                    "preprocessor",
                    eval_preprocessor,
                ),
                (
                    "model",
                    fresh_xgb(),
                ),
            ])

            start = time.perf_counter()

            eval_pipeline.fit(
                X_train[selected],
                y_train,
            )

            y_pred = eval_pipeline.predict(
                X_test[selected]
            )

            y_score = eval_pipeline.predict_proba(
                X_test[selected]
            )[:, 1]

            runtime = (
                time.perf_counter()
                - start
            )

            metric_values = calculate_metrics(
                y_test,
                y_pred,
                y_score,
            )

            transformed_count = len(
                eval_pipeline
                .named_steps["preprocessor"]
                .get_feature_names_out()
            )

            performance_rows.append({
                "dataset": dataset_name,
                "run_id": run_id,
                "repeat": split_info["repeat"],
                "fold": split_info["fold"],
                "selector": selector_name,
                "subset": subset_label,
                "subset_fraction": fraction,
                "selected_source_features": n_select,
                "selected_transformed_features": transformed_count,
                "feature_reduction_pct": reduction_pct,
                "selector_runtime_seconds_for_fold": selector_runtime,
                "xgb_runtime_seconds": runtime,
                **metric_values,
            })

            print(
                f"{selector_name:14s} "
                f"{subset_label}: "
                f"ROC={metric_values['roc_auc']:.4f} "
                f"MCC={metric_values['mcc']:.4f} "
                f"F1={metric_values['f1_adverse']:.4f}"
            )


rankings_all = pd.DataFrame(
    ranking_rows
)

selected_sets = pd.DataFrame(
    selection_rows
)

fs_fold_results = pd.DataFrame(
    performance_rows
)

rankings_all.to_csv(
    OUT_DIR
    / "german_source_feature_rankings_all.csv",
    index=False,
)

selected_sets.to_csv(
    OUT_DIR
    / "german_selected_feature_sets_all.csv",
    index=False,
)

fs_fold_results.to_csv(
    OUT_DIR
    / "german_feature_selection_fold_results.csv",
    index=False,
)

print(
    "\nGerman feature-selection development "
    "runs completed."
)



R1_F1 (1/25)
Selector rankings completed in 0.89s
Chi2           Top25: ROC=0.7530 MCC=0.2741 F1=0.4272
Chi2           Top50: ROC=0.8134 MCC=0.4379 F1=0.6000
Chi2           Top75: ROC=0.8350 MCC=0.4989 F1=0.6446
MI             Top25: ROC=0.7780 MCC=0.3659 F1=0.5500
MI             Top50: ROC=0.7972 MCC=0.3796 F1=0.5620
MI             Top75: ROC=0.8382 MCC=0.4840 F1=0.6271
L1             Top25: ROC=0.7481 MCC=0.3695 F1=0.5574
L1             Top50: ROC=0.7972 MCC=0.3695 F1=0.5574
L1             Top75: ROC=0.8203 MCC=0.5247 F1=0.6667
RF             Top25: ROC=0.6880 MCC=0.1749 F1=0.3529
RF             Top50: ROC=0.7986 MCC=0.4040 F1=0.5536
RF             Top75: ROC=0.8326 MCC=0.4931 F1=0.6195
RFE            Top25: ROC=0.7402 MCC=0.2555 F1=0.4706
RFE            Top50: ROC=0.8094 MCC=0.4138 F1=0.5984
RFE            Top75: ROC=0.8103 MCC=0.4751 F1=0.6281
EqualConsensus Top25: ROC=0.7481 MCC=0.3695 F1=0.5574
EqualConsensus Top50: ROC=0.8097 MCC=0.4619 F1=0.6167
EqualConsensus Top75: ROC=0.813

## 9. Compute source-feature subset stability

In [11]:

def parse_feature_set(text):
    if pd.isna(text) or text == "":
        return set()

    return set(
        str(text).split(";")
    )


stability_rows = []

for (
    selector_name,
    subset_label,
), group in selected_sets.groupby(
    ["selector", "subset"]
):
    sets = [
        parse_feature_set(value)
        for value in group[
            "selected_features"
        ]
    ]

    pair_scores = []

    for set_a, set_b in combinations(
        sets,
        2,
    ):
        union = set_a | set_b

        jaccard = (
            len(set_a & set_b)
            / len(union)
            if union
            else 1.0
        )

        pair_scores.append(
            jaccard
        )

    stability_rows.append({
        "selector": selector_name,
        "subset": subset_label,
        "selected_source_features": int(
            group[
                "selected_source_features"
            ].iloc[0]
        ),
        "pairwise_comparisons": len(
            pair_scores
        ),
        "mean_jaccard": np.mean(
            pair_scores
        ),
        "std_jaccard": np.std(
            pair_scores,
            ddof=1,
        ),
        "min_jaccard": np.min(
            pair_scores
        ),
        "max_jaccard": np.max(
            pair_scores
        ),
    })


stability = pd.DataFrame(
    stability_rows
)

stability.to_csv(
    OUT_DIR
    / "german_source_feature_stability.csv",
    index=False,
)

display(
    stability.sort_values(
        [
            "subset",
            "mean_jaccard",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


,selector,subset,selected_source_features,pairwise_comparisons,mean_jaccard,std_jaccard,min_jaccard,max_jaccard
12,RF,Top25,5,300,1.000000,0.000000,1.000000,1.0
0,Chi2,Top25,5,300,0.759048,0.174712,0.428571,1.0
3,EqualConsensus,Top25,5,300,0.671190,0.182098,0.250000,1.0
6,L1,Top25,5,300,0.590317,0.171440,0.250000,1.0
15,RFE,Top25,5,300,0.506892,0.170803,0.111111,1.0
9,MI,Top25,5,300,0.478095,0.164705,0.250000,1.0
13,RF,Top50,10,300,0.872020,0.088079,0.666667,1.0
1,Chi2,Top50,10,300,0.769946,0.105747,0.538462,1.0
16,RFE,Top50,10,300,0.731312,0.138824,0.428571,1.0
7,L1,Top50,10,300,0.678628,0.121241,0.428571,1.0


## 10. Compute source-feature selection frequencies

In [12]:

frequency_rows = []

for (
    selector_name,
    subset_label,
), group in selected_sets.groupby(
    ["selector", "subset"]
):
    parsed_sets = [
        parse_feature_set(value)
        for value in group[
            "selected_features"
        ]
    ]

    for feature in source_features:
        count = sum(
            feature in feature_set
            for feature_set in parsed_sets
        )

        frequency_rows.append({
            "selector": selector_name,
            "subset": subset_label,
            "source_feature": feature,
            "selected_runs": count,
            "total_runs": len(
                parsed_sets
            ),
            "selection_frequency": (
                count
                / len(parsed_sets)
            ),
        })


selection_frequency = pd.DataFrame(
    frequency_rows
)

selection_frequency.to_csv(
    OUT_DIR
    / "german_source_feature_selection_frequency.csv",
    index=False,
)

display(
    selection_frequency[
        (
            selection_frequency["selector"]
            == "EqualConsensus"
        )
        & (
            selection_frequency["subset"]
            == "Top50"
        )
    ]
    .sort_values(
        "selection_frequency",
        ascending=False,
    )
)


,selector,subset,source_feature,selected_runs,total_runs,selection_frequency
80,EqualConsensus,Top50,status,25,25,1.00
81,EqualConsensus,Top50,credit_history,25,25,1.00
93,EqualConsensus,Top50,duration,25,25,1.00
94,EqualConsensus,Top50,amount,25,25,1.00
83,EqualConsensus,Top50,savings,24,25,0.96
95,EqualConsensus,Top50,installment_rate,23,25,0.92
92,EqualConsensus,Top50,foreign_worker,22,25,0.88
89,EqualConsensus,Top50,housing,20,25,0.80
88,EqualConsensus,Top50,other_installment_plans,16,25,0.64
87,EqualConsensus,Top50,property,11,25,0.44


## 11. Create performance summary

In [13]:

summary = (
    fs_fold_results
    .groupby(
        [
            "selector",
            "subset",
            "subset_fraction",
            "selected_source_features",
        ],
        as_index=False,
    )
    .agg(
        ROC_AUC_mean=("roc_auc", "mean"),
        ROC_AUC_std=("roc_auc", "std"),
        PR_AUC_mean=("pr_auc", "mean"),
        PR_AUC_std=("pr_auc", "std"),
        Recall_mean=("recall_adverse", "mean"),
        Precision_mean=("precision_adverse", "mean"),
        F1_mean=("f1_adverse", "mean"),
        Balanced_Accuracy_mean=("balanced_accuracy", "mean"),
        MCC_mean=("mcc", "mean"),
        MCC_std=("mcc", "std"),
        GMean_mean=("gmean", "mean"),
        KS_mean=("ks_statistic", "mean"),
        XGB_Runtime_mean=("xgb_runtime_seconds", "mean"),
    )
)

summary = summary.merge(
    stability[
        [
            "selector",
            "subset",
            "mean_jaccard",
            "std_jaccard",
        ]
    ],
    on=[
        "selector",
        "subset",
    ],
    how="left",
)

summary["feature_reduction_pct"] = (
    100.0
    * (
        1.0
        - summary[
            "selected_source_features"
        ]
        / len(source_features)
    )
)

summary.to_csv(
    OUT_DIR
    / "german_feature_selection_summary.csv",
    index=False,
)

display(
    summary.sort_values(
        [
            "MCC_mean",
            "ROC_AUC_mean",
        ],
        ascending=False,
    )
)


,selector,subset,subset_fraction,selected_source_features,ROC_AUC_mean,ROC_AUC_std,PR_AUC_mean,PR_AUC_std,Recall_mean,Precision_mean,F1_mean,Balanced_Accuracy_mean,MCC_mean,MCC_std,GMean_mean,KS_mean,XGB_Runtime_mean,mean_jaccard,std_jaccard,feature_reduction_pct
2,Chi2,Top75,0.75,15,0.788434,0.022248,0.622283,0.055999,0.490576,0.632947,0.547579,0.683953,0.398727,0.051125,0.653594,0.485773,0.352342,0.886667,0.075850,25.0
5,EqualConsensus,Top75,0.75,15,0.781792,0.020931,0.617878,0.056087,0.482097,0.634873,0.544830,0.681464,0.396326,0.062893,0.650017,0.469926,0.347381,0.812925,0.075860,25.0
17,RFE,Top75,0.75,15,0.782189,0.020101,0.613742,0.055288,0.484674,0.632635,0.545826,0.681847,0.395844,0.056642,0.651242,0.473125,0.343510,0.798302,0.081928,25.0
11,MI,Top75,0.75,15,0.785023,0.026057,0.621040,0.052186,0.489509,0.628547,0.547716,0.682804,0.395472,0.074088,0.653259,0.476830,0.346069,0.763574,0.080928,25.0
1,Chi2,Top50,0.50,10,0.775373,0.022399,0.607207,0.050728,0.484768,0.622478,0.541589,0.679324,0.387855,0.052769,0.649064,0.461149,0.324551,0.769946,0.105747,50.0
8,L1,Top75,0.75,15,0.784249,0.023652,0.614700,0.059960,0.473351,0.620457,0.533829,0.674725,0.380692,0.067859,0.642037,0.476189,0.337088,0.761048,0.075711,25.0
7,L1,Top50,0.50,10,0.768798,0.023385,0.593123,0.058765,0.482001,0.606604,0.532552,0.673213,0.372502,0.057227,0.643228,0.458546,0.317725,0.678628,0.121241,50.0
16,RFE,Top50,0.50,10,0.765458,0.025256,0.600597,0.049873,0.474433,0.608715,0.529401,0.671142,0.370553,0.053051,0.639597,0.444495,0.314085,0.731312,0.138824,50.0
14,RF,Top75,0.75,15,0.773140,0.028755,0.601511,0.069894,0.465731,0.605907,0.521978,0.668210,0.365487,0.093905,0.632911,0.450025,0.337051,0.877206,0.064634,25.0
4,EqualConsensus,Top50,0.50,10,0.764991,0.024262,0.595878,0.054277,0.470385,0.602913,0.525367,0.668811,0.364991,0.050829,0.636922,0.448184,0.318307,0.678231,0.114828,50.0


## 12. Paired deltas against the Step-3 all-feature XGBoost baseline

In [14]:

baseline_xgb = baseline_results[
    (
        baseline_results["dataset"]
        == dataset_name
    )
    & (
        baseline_results["model"]
        == "XGB"
    )
][
    [
        "run_id",
        "roc_auc",
        "pr_auc",
        "recall_adverse",
        "f1_adverse",
        "balanced_accuracy",
        "mcc",
        "gmean",
        "ks_statistic",
    ]
].copy()

baseline_xgb = baseline_xgb.rename(
    columns={
        column: (
            "baseline_" + column
            if column != "run_id"
            else column
        )
        for column in baseline_xgb.columns
    }
)

paired = fs_fold_results.merge(
    baseline_xgb,
    on="run_id",
    how="left",
    validate="many_to_one",
)

for metric in [
    "roc_auc",
    "pr_auc",
    "recall_adverse",
    "f1_adverse",
    "balanced_accuracy",
    "mcc",
    "gmean",
    "ks_statistic",
]:
    paired[
        "delta_" + metric
    ] = (
        paired[metric]
        - paired[
            "baseline_" + metric
        ]
    )

paired.to_csv(
    OUT_DIR
    / "german_feature_selection_paired_deltas.csv",
    index=False,
)

delta_summary = (
    paired.groupby(
        [
            "selector",
            "subset",
        ],
        as_index=False,
    )
    .agg(
        Delta_ROC_AUC=("delta_roc_auc", "mean"),
        Delta_PR_AUC=("delta_pr_auc", "mean"),
        Delta_Recall=("delta_recall_adverse", "mean"),
        Delta_F1=("delta_f1_adverse", "mean"),
        Delta_BalAcc=("delta_balanced_accuracy", "mean"),
        Delta_MCC=("delta_mcc", "mean"),
        Delta_GMean=("delta_gmean", "mean"),
        Delta_KS=("delta_ks_statistic", "mean"),
    )
)

delta_summary.to_csv(
    OUT_DIR
    / "german_feature_selection_delta_summary.csv",
    index=False,
)

display(
    delta_summary.sort_values(
        "Delta_MCC",
        ascending=False,
    )
)


,selector,subset,Delta_ROC_AUC,Delta_PR_AUC,Delta_Recall,Delta_F1,Delta_BalAcc,Delta_MCC,Delta_GMean,Delta_KS
2,Chi2,Top75,-0.001935,-0.010083,0.012238,0.003245,0.002277,-0.000094,0.005086,0.007002
5,EqualConsensus,Top75,-0.008577,-0.014488,0.003759,0.000496,-0.000212,-0.002495,0.001508,-0.008845
17,RFE,Top75,-0.008179,-0.018623,0.006336,0.001492,0.000171,-0.002977,0.002734,-0.005646
11,MI,Top75,-0.005346,-0.011326,0.011171,0.003382,0.001128,-0.003350,0.004750,-0.001941
1,Chi2,Top50,-0.014996,-0.025159,0.006430,-0.002745,-0.002353,-0.010967,0.000555,-0.017622
8,L1,Top75,-0.006120,-0.017666,-0.004987,-0.010505,-0.006951,-0.018130,-0.006472,-0.002582
7,L1,Top50,-0.021571,-0.039242,0.003663,-0.011782,-0.008463,-0.026319,-0.005281,-0.020225
16,RFE,Top50,-0.024911,-0.031769,-0.003905,-0.014933,-0.010534,-0.028268,-0.008911,-0.034276
14,RF,Top75,-0.017228,-0.030855,-0.012608,-0.022356,-0.013466,-0.033334,-0.015597,-0.028746
4,EqualConsensus,Top50,-0.025378,-0.036487,-0.007953,-0.018967,-0.012865,-0.033830,-0.011587,-0.030587


## 13. Generate a development decision table

This table is **not an automatic winner declaration**. It is only a compact decision aid. The final frozen configuration will be chosen after reviewing:

- ROC-AUC preservation
- PR-AUC
- adverse-class F1/recall
- MCC and balanced accuracy
- source-feature reduction
- Jaccard stability

Australian and Taiwan performance must not influence this decision.


In [15]:

decision_table = summary.merge(
    delta_summary,
    on=[
        "selector",
        "subset",
    ],
    how="left",
)

decision_table = decision_table[
    [
        "selector",
        "subset",
        "selected_source_features",
        "feature_reduction_pct",
        "ROC_AUC_mean",
        "PR_AUC_mean",
        "Recall_mean",
        "F1_mean",
        "Balanced_Accuracy_mean",
        "MCC_mean",
        "mean_jaccard",
        "Delta_ROC_AUC",
        "Delta_PR_AUC",
        "Delta_Recall",
        "Delta_F1",
        "Delta_BalAcc",
        "Delta_MCC",
    ]
]

decision_table.to_csv(
    OUT_DIR
    / "german_feature_selection_decision_table.csv",
    index=False,
)

display(
    decision_table.sort_values(
        [
            "MCC_mean",
            "ROC_AUC_mean",
            "mean_jaccard",
        ],
        ascending=False,
    )
)


,selector,subset,selected_source_features,feature_reduction_pct,ROC_AUC_mean,PR_AUC_mean,Recall_mean,F1_mean,Balanced_Accuracy_mean,MCC_mean,mean_jaccard,Delta_ROC_AUC,Delta_PR_AUC,Delta_Recall,Delta_F1,Delta_BalAcc,Delta_MCC
2,Chi2,Top75,15,25.0,0.788434,0.622283,0.490576,0.547579,0.683953,0.398727,0.886667,-0.001935,-0.010083,0.012238,0.003245,0.002277,-0.000094
5,EqualConsensus,Top75,15,25.0,0.781792,0.617878,0.482097,0.544830,0.681464,0.396326,0.812925,-0.008577,-0.014488,0.003759,0.000496,-0.000212,-0.002495
17,RFE,Top75,15,25.0,0.782189,0.613742,0.484674,0.545826,0.681847,0.395844,0.798302,-0.008179,-0.018623,0.006336,0.001492,0.000171,-0.002977
11,MI,Top75,15,25.0,0.785023,0.621040,0.489509,0.547716,0.682804,0.395472,0.763574,-0.005346,-0.011326,0.011171,0.003382,0.001128,-0.003350
1,Chi2,Top50,10,50.0,0.775373,0.607207,0.484768,0.541589,0.679324,0.387855,0.769946,-0.014996,-0.025159,0.006430,-0.002745,-0.002353,-0.010967
8,L1,Top75,15,25.0,0.784249,0.614700,0.473351,0.533829,0.674725,0.380692,0.761048,-0.006120,-0.017666,-0.004987,-0.010505,-0.006951,-0.018130
7,L1,Top50,10,50.0,0.768798,0.593123,0.482001,0.532552,0.673213,0.372502,0.678628,-0.021571,-0.039242,0.003663,-0.011782,-0.008463,-0.026319
16,RFE,Top50,10,50.0,0.765458,0.600597,0.474433,0.529401,0.671142,0.370553,0.731312,-0.024911,-0.031769,-0.003905,-0.014933,-0.010534,-0.028268
14,RF,Top75,15,25.0,0.773140,0.601511,0.465731,0.521978,0.668210,0.365487,0.877206,-0.017228,-0.030855,-0.012608,-0.022356,-0.013466,-0.033334
4,EqualConsensus,Top50,10,50.0,0.764991,0.595878,0.470385,0.525367,0.668811,0.364991,0.678231,-0.025378,-0.036487,-0.007953,-0.018967,-0.012865,-0.033830


## 14. Final consistency checks and manifest

In [16]:

# 6 selectors x 3 subset levels x 25 outer runs
expected_performance_rows = (
    len(SELECTOR_NAMES)
    * len(SUBSET_FRACTIONS)
    * 25
)

assert len(fs_fold_results) == expected_performance_rows, (
    f"Expected {expected_performance_rows} rows, "
    f"found {len(fs_fold_results)}."
)

assert (
    fs_fold_results[
        "roc_auc"
    ].between(0, 1).all()
)

assert (
    fs_fold_results[
        "pr_auc"
    ].between(0, 1).all()
)

assert (
    fs_fold_results[
        "mcc"
    ].between(-1, 1).all()
)

assert len(rankings_all) == (
    25
    * len(SELECTOR_NAMES)
    * len(source_features)
)

assert len(selected_sets) == (
    25
    * len(SELECTOR_NAMES)
    * len(SUBSET_FRACTIONS)
)

config = {
    "stage": (
        "Objective 3 Step 4 - German development"
    ),
    "development_dataset": "German Credit",
    "external_datasets_not_used_for_selection": [
        "Australian Credit Approval",
        "Taiwan Credit Card Default",
    ],
    "source_predictors": len(
        source_features
    ),
    "selectors": SELECTOR_NAMES,
    "subset_fractions": SUBSET_FRACTIONS,
    "evaluation_classifier": (
        "Fixed Step-3 XGBoost configuration"
    ),
    "outer_validation": (
        "Same StratifiedGroupKFold "
        "5x5 folds as Step 3"
    ),
    "group_score_rule": (
        "Mean normalized transformed-variable "
        "relevance within each original source feature"
    ),
    "consensus_rule": (
        "Equal mean of five normalized "
        "source-feature selector ranks"
    ),
    "imbalance_treatment": "none",
    "hyperparameter_tuning": "none",
    "calibration": "none",
    "threshold_optimization": "none",
    "final_configuration_selection": (
        "Manual review after Step-4 results; "
        "must use German evidence only"
    ),
}

with open(
    OUT_DIR
    / "step4_experiment_configuration.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        config,
        file,
        indent=4,
    )

manifest = []

for path in sorted(
    OUT_DIR.iterdir()
):
    if path.is_file():
        manifest.append(
            path.name
        )

pd.DataFrame(
    {
        "generated_file": manifest
    }
).to_csv(
    OUT_DIR
    / "step4_output_manifest.csv",
    index=False,
)

print("=" * 80)
print("STEP 4 COMPLETED SUCCESSFULLY")
print("=" * 80)
print("\nOutput folder:")
print(OUT_DIR)
print("\nMost important files:")
print(" - german_feature_selection_decision_table.csv")
print(" - german_feature_selection_summary.csv")
print(" - german_source_feature_stability.csv")
print(" - german_source_feature_selection_frequency.csv")
print(" - german_feature_selection_paired_deltas.csv")


STEP 4 COMPLETED SUCCESSFULLY

Output folder:
D:\PHD\Research Paper writing\3rd Obj. paper\results\feature_selection_german_development

Most important files:
 - german_feature_selection_decision_table.csv
 - german_feature_selection_summary.csv
 - german_source_feature_stability.csv
 - german_source_feature_selection_frequency.csv
 - german_feature_selection_paired_deltas.csv
